In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import Optional, Dict, List
import csv
import random

In [25]:
def find_appid(game_name: str, lookup_df: pd.DataFrame) -> Optional[int]:
    """
    Find AppID from the lookup file by matching game name
    """
    game_name_clean = game_name.strip()
    
    # Try exact match first (case insensitive)
    exact_match = lookup_df[lookup_df['name'].str.lower() == game_name_clean.lower()]
    if not exact_match.empty:
        return int(exact_match.iloc[0]['appid'])
    
    # Try partial match
    partial_match = lookup_df[lookup_df['name'].str.contains(game_name_clean, case=False, na=False, regex=False)]
    if not partial_match.empty:
        # Prefer closer matches
        for idx, row in partial_match.iterrows():
            if game_name_clean.lower() in row['name'].lower():
                return int(row['appid'])
        return int(partial_match.iloc[0]['appid'])
    
    return None

In [26]:
def scrape_steam_charts(app_id):
    url = f"https://steamcharts.com/app/{app_id}"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers = headers)

    if response.status_code!= 200:
        print(f'Status Code: {response.status_code}')

    soup = BeautifulSoup(response.text, 'html.parser')

    table = soup.find('table', {'class': 'common-table'})

    if table is None:
        print(f"Warning: No table found for App ID {app_id}")
        return None  # Return None so your main loop can skip this game
    data = []

    for row in table.find_all('tr')[1:]:
        cols = row.find_all('td')
        

        if len(cols) >= 3:
            month_year = cols[0].text.strip()
            avg_players = cols[1].text.strip().replace(',', '') 
            peak_players = cols[2].text.strip().replace(',', '')
            
            data.append([month_year, avg_players, peak_players])

    df = pd.DataFrame(data, columns=['Month-Year', 'Avg_Players', 'Peak_Players'])
    return df


In [27]:

lookup_df = pd.read_csv("complete_steam_lookup_2026.csv")
df_list = []
with open('game_list.csv', mode = 'r', newline = '') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        app_id = find_appid(row[0], lookup_df)
        df = scrape_steam_charts(app_id)
        if df is not None:
            df['Game'] = row[0]
            df = df[['Game', 'Month-Year', 'Avg_Players']][1:]
            df_list.append(df)
        else:
            print('Not Available')

        time.sleep(random.uniform(2, 5))

full_df = pd.concat(df_list, axis = 0, ignore_index = True)
full_df['Month-Year'] = pd.to_datetime(full_df['Month-Year'], format='%B %Y', errors='coerce')
full_df = full_df.rename(columns = {'Month-Year':'Date'})
full_df.to_csv('monthy_player_count.csv')


In [ ]:
import pandas as pd

original_games = pd.read_csv('game_list.csv')

input_names = set(original_games['Game'].unique())
scraped_names = set(full_df['Game'].unique())

missing_games = input_names - scraped_names

print(f"Total Games in list: {len(input_names)}")
print(f"Successfully Scraped: {len(scraped_names)}")
print(f"Missing Games: {len(missing_games)}")

if missing_games:
    print("\n--- Missing Games List ---")
    for game in sorted(missing_games):
        print(f"- {game}")

Total Games in list: 50
Successfully Scraped: 48
Missing Games: 2

--- Missing Games List ---
- Crosshair X
- Rocket League®
